# Notebook 4: Walk-Forward Backtesting
**AdaptiveBeta — AI-Powered Portfolio Optimisation**  
**Author:** Kunal | M.Tech AI & ML, Symbiosis Institute of Technology, Pune

This is the central results notebook. It:
1. Runs a walk-forward backtest (2018–2024, out-of-sample)
2. Compares 5 strategies with realistic transaction costs
3. Computes comprehensive performance metrics
4. Analyses stress events (COVID, ADANI crisis, IL&FS)
5. Runs ablation studies to isolate each component's contribution
6. Generates QuantStats tearsheet

**Walk-Forward Config:**
- Train: 3 years rolling | Test: 1 year | Step: 1 year
- First OOS period: 2018 (trained on 2015–2017)
- Transaction costs: 0.08% round-trip per rebalance

In [ ]:
import sys, os, json, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/AI_Finance_Project'

REPO = '/content/drive/MyDrive/AI_Finance_Project/repo'
if os.path.exists(REPO):
    sys.path.insert(0, REPO)

print('Drive mounted.')

## 4.1 — Load All Data & Models

In [ ]:
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler

TICKERS = [
    'RELIANCE.NS','TCS.NS','HDFCBANK.NS','INFY.NS','ICICIBANK.NS',
    'HINDUNILVR.NS','ITC.NS','SBIN.NS','BHARTIARTL.NS','KOTAKBANK.NS',
    'LT.NS','AXISBANK.NS','ASIANPAINT.NS','MARUTI.NS','BAJFINANCE.NS',
    'HCLTECH.NS','SUNPHARMA.NS','TITAN.NS','ULTRACEMCO.NS','NESTLEIND.NS',
    'WIPRO.NS','POWERGRID.NS','NTPC.NS','ONGC.NS','TECHM.NS',
    'JSWSTEEL.NS','TATASTEEL.NS','M&M.NS','ADANIENT.NS','ADANIPORTS.NS',
    'COALINDIA.NS','BAJAJFINSV.NS','HDFCLIFE.NS','SBILIFE.NS','DRREDDY.NS',
    'DIVISLAB.NS','CIPLA.NS','EICHERMOT.NS','HEROMOTOCO.NS','APOLLOHOSP.NS',
    'BAJAJ-AUTO.NS','BRITANNIA.NS','GRASIM.NS','INDUSINDBK.NS','TATACONSUM.NS',
    'UPL.NS','BPCL.NS','IOC.NS','HINDALCO.NS',
]

STRATEGIES = ['adaptive_beta', 'static_capm_mvo', 'kalman_mvo', 'equal_weight', 'buy_hold_nifty']

# Transaction costs
BROKERAGE = 0.0005
SLIPPAGE  = 0.0003
ROUND_TRIP_COST = 2 * (BROKERAGE + SLIPPAGE)  # 0.16% per rebalance

# Signal config
with open(f'{ROOT}/models/signal_config.json') as f:
    sig_cfg = json.load(f)
THRESHOLD     = sig_cfg['betavol_threshold']
VIX_THRESHOLD = sig_cfg['vix_threshold']
TRAIN_END     = sig_cfg['train_end']
TEST_START    = sig_cfg['test_start']
print(f'BetaVol threshold: {THRESHOLD:.4f}')
print(f'VIX threshold: {VIX_THRESHOLD}')

# Load prices
prices_df = pd.read_csv(f'{ROOT}/raw_data/stocks/all_stocks_prices.csv',
                         parse_dates=['date']).set_index('date').sort_index()
nifty_df  = pd.read_csv(f'{ROOT}/raw_data/market/nifty50.csv',
                         parse_dates=['date']).set_index('date').sort_index()
vix_df    = pd.read_csv(f'{ROOT}/raw_data/market/india_vix.csv',
                         parse_dates=['date']).set_index('date').sort_index()

avail_tickers = [t for t in TICKERS if t in prices_df.columns]
prices    = prices_df[avail_tickers]
nifty_ret = np.log(nifty_df['Close'] / nifty_df['Close'].shift(1))
vix_s     = vix_df['Close'].reindex(nifty_ret.index).ffill()

# Load betas
betas_60   = pd.read_csv(f'{ROOT}/features/beta60d.csv',
                          parse_dates=['date']).set_index('date')
betavol_60 = pd.read_csv(f'{ROOT}/features/betavol_60d.csv',
                          parse_dates=['date']).set_index('date')

# Load Kalman betas
kalman_beta_path = f'{ROOT}/features/kalman_betas.csv'
kalman_betas = None
if os.path.exists(kalman_beta_path):
    kalman_betas = pd.read_csv(kalman_beta_path, parse_dates=['date']).set_index('date')
    print(f'Kalman betas loaded: {kalman_betas.shape}')

# Load HMM regimes
regime_labels = pd.read_csv(f'{ROOT}/features/hmm_regimes.csv',
                             parse_dates=['date']).set_index('date')['regime']

# Load stacked features for LSTM prediction
stacked = pd.read_csv(f'{ROOT}/features/stacked_features.csv',
                       parse_dates=['date']).set_index('date')
feature_cols = [c for c in stacked.columns if c not in ['target', 'ticker']]

print(f'Prices: {prices.shape}')
print(f'Betas 60d: {betas_60.shape}')
print(f'Regimes: {regime_labels.value_counts().to_dict()}')

## 4.2 — LSTM Class & Helper Functions

In [ ]:
SEQ_LEN = 30

class BetaLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=64, n_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, n_layers, batch_first=True,
                             dropout=dropout if n_layers > 1 else 0.0)
        self.norm = nn.LayerNorm(hidden_size)
        self.head = nn.Sequential(
            nn.Linear(hidden_size, 32), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(32, 1)
        )
    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.norm(out[:, -1, :])
        return self.head(out).squeeze(-1)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def load_lstm(path, n_features):
    """Load saved LSTM checkpoint."""
    m = BetaLSTM(n_features).to(device)
    m.load_state_dict(torch.load(path, map_location=device))
    m.eval()
    return m

def lstm_predict_scalar(model, scaler, date):
    """Get single-point LSTM prediction of portfolio betavol for a date."""
    try:
        mask = stacked.index <= date
        by_date = stacked[mask].groupby(stacked[mask].index)[feature_cols].mean()
        if len(by_date) < SEQ_LEN:
            return float(THRESHOLD)
        X_seq = by_date.tail(SEQ_LEN).values
        X_sc  = scaler.transform(X_seq)
        X_t   = torch.FloatTensor(X_sc).unsqueeze(0).to(device)
        with torch.no_grad():
            pred = model(X_t).item()
        return max(0, pred)  # betavol cannot be negative
    except Exception:
        return float(THRESHOLD)

def transaction_cost(old_w, new_w):
    all_t = old_w.index.union(new_w.index)
    turnover = (new_w.reindex(all_t).fillna(0) - old_w.reindex(all_t).fillna(0)).abs().sum() / 2
    return float(turnover * ROUND_TRIP_COST)

def get_signal(pred_bv, vix_lvl):
    if vix_lvl > VIX_THRESHOLD: return 'MIN_VARIANCE'
    if pred_bv > THRESHOLD:     return 'REBALANCE'
    return 'HOLD'

def get_mode(signal, regime):
    if signal == 'MIN_VARIANCE': return 'min_variance'
    if signal == 'REBALANCE':
        if regime == 'bull': return 'max_sharpe_beta_constrained'
        if regime == 'bear': return 'min_variance'
        return 'risk_parity'
    return 'hold'

print(f'Device: {device}')
print('Helper functions defined.')

## 4.3 — Inline Optimiser (for backtest speed)

In [ ]:
from pypfopt import EfficientFrontier, risk_models, expected_returns

def optimise(prices_w, betas_s, mode, beta_target=0.85, max_w=0.15, rf=0.065):
    n = len(prices_w.columns)
    tickers = prices_w.columns.tolist()
    S = risk_models.CovarianceShrinkage(prices_w).ledoit_wolf()

    try:
        if mode == 'max_sharpe_beta_constrained':
            mu = expected_returns.mean_historical_return(prices_w, frequency=252)
            betas_arr = betas_s.reindex(tickers).fillna(1.0).values
            ef = EfficientFrontier(mu, S)
            ef.add_constraint(lambda w: w >= 0)
            ef.add_constraint(lambda w: w <= max_w)
            ef.add_constraint(lambda w: sum(w[i]*betas_arr[i] for i in range(n)) <= beta_target)
            ef.add_constraint(lambda w: sum(w[i]*betas_arr[i] for i in range(n)) >= beta_target-0.2)
            ef.max_sharpe(risk_free_rate=rf)
            return pd.Series(ef.clean_weights())

        if mode in ('min_variance', 'risk_parity'):
            ef = EfficientFrontier(None, S)
            ef.add_constraint(lambda w: w >= 0)
            ef.add_constraint(lambda w: w <= max_w)
            ef.min_volatility()
            return pd.Series(ef.clean_weights())

    except Exception:
        pass

    return pd.Series({t: 1/n for t in tickers})

print('Optimiser ready.')

## 4.4 — Walk-Forward Backtest Engine

In [ ]:
# Load the globally trained LSTM + scaler for simplified backtest
# (In a full walk-forward, you'd retrain per fold — use this for speed)
global_scaler = pickle.load(open(f'{ROOT}/models/scaler.pkl', 'rb'))
n_features    = len(feature_cols)
lstm_model    = load_lstm(f'{ROOT}/models/lstm_best.pt', n_features)
print(f'LSTM loaded: {n_features} features')

# Walk-forward parameters
TRAIN_YEARS = 3
TEST_YEARS  = 1
START_YEAR  = 2015
END_YEAR    = 2024

fold_starts = pd.date_range(
    start=f'{START_YEAR + TRAIN_YEARS}-01-01',
    end=f'{END_YEAR}-01-01',
    freq='YS'
)

print(f'Walk-forward folds: {len(fold_starts)}')
for fs in fold_starts:
    fe = fs + pd.DateOffset(years=TEST_YEARS) - pd.Timedelta(days=1)
    ts = fs - pd.DateOffset(years=TRAIN_YEARS)
    print(f'  Train: {ts.date()} → {fs.date()} | Test: {fs.date()} → {fe.date()}')

In [ ]:
# Full walk-forward backtest
all_returns = {s: [] for s in STRATEGIES}
rebalance_log = []

equal_weights = pd.Series({t: 1/len(avail_tickers) for t in avail_tickers})

for fold_num, fold_start in enumerate(fold_starts):
    fold_end   = fold_start + pd.DateOffset(years=TEST_YEARS) - pd.Timedelta(days=1)

    print(f'\n=== Fold {fold_num+1}: {fold_start.date()} → {fold_end.date()} ===')

    # Test period dates
    test_idx  = prices.index[(prices.index >= fold_start) & (prices.index <= fold_end)]
    if len(test_idx) < 2:
        print('  Insufficient data — skipping')
        continue

    # Initial weights for this fold
    weights = {s: equal_weights.copy() for s in STRATEGIES}

    for i in range(1, len(test_idx)):
        date      = test_idx[i]
        prev_date = test_idx[i - 1]

        # Daily stock returns
        p_now  = prices.loc[date].reindex(avail_tickers).ffill()
        p_prev = prices.loc[prev_date].reindex(avail_tickers).ffill()
        stk_ret = np.log((p_now / p_prev).replace([np.inf, -np.inf], 0)).fillna(0)

        # Buy & hold NIFTY
        nifty_r = float(nifty_ret.loc[date]) if date in nifty_ret.index else 0.0
        all_returns['buy_hold_nifty'].append({'date': date, 'ret': nifty_r})

        # Equal weight (monthly rebalance)
        if date.day <= 5:
            weights['equal_weight'] = equal_weights.copy()
        r_ew = float((weights['equal_weight'] * stk_ret.reindex(weights['equal_weight'].index).fillna(0)).sum())
        all_returns['equal_weight'].append({'date': date, 'ret': r_ew})

        # ML strategies — check every 5 days
        should_check = (i % 5 == 0)

        if should_check:
            pw      = prices.loc[:prev_date].tail(252)
            pw      = pw.dropna(axis=1, how='any')
            cur_tkrs = pw.columns.tolist()

            cur_betas = (betas_60.loc[prev_date].reindex(cur_tkrs).fillna(1.0)
                         if prev_date in betas_60.index else pd.Series(1.0, index=cur_tkrs))
            vix_now   = float(vix_s.loc[prev_date]) if prev_date in vix_s.index else 20.0
            regime_now = str(regime_labels.loc[prev_date]) if prev_date in regime_labels.index else 'transition'

            # --- Adaptive Beta ---
            pred_bv = lstm_predict_scalar(lstm_model, global_scaler, prev_date)
            sig     = get_signal(pred_bv, vix_now)
            mode    = get_mode(sig, regime_now)

            if mode != 'hold' and len(pw) >= 60:
                new_w = optimise(pw, cur_betas, mode, beta_target=max(0.6, pred_bv * 0.8))
                cost  = transaction_cost(weights['adaptive_beta'].reindex(cur_tkrs).fillna(0), new_w)
                weights['adaptive_beta'] = new_w
                all_returns['adaptive_beta'].append({'date': date, 'ret': -cost})
                rebalance_log.append({'date': date, 'signal': sig, 'mode': mode,
                                       'regime': regime_now, 'pred_bv': pred_bv, 'cost': cost})
                r_ab = float((new_w * stk_ret.reindex(new_w.index).fillna(0)).sum())
            else:
                r_ab = float((weights['adaptive_beta'] * stk_ret.reindex(weights['adaptive_beta'].index).fillna(0)).sum())
            all_returns['adaptive_beta'].append({'date': date, 'ret': r_ab})

            # --- Static CAPM MVO (monthly) ---
            if date.day <= 5 and len(pw) >= 60:
                new_w = optimise(pw, cur_betas, 'max_sharpe_beta_constrained')
                cost  = transaction_cost(weights['static_capm_mvo'].reindex(cur_tkrs).fillna(0), new_w)
                weights['static_capm_mvo'] = new_w
                all_returns['static_capm_mvo'].append({'date': date, 'ret': -cost})

            r_sc = float((weights['static_capm_mvo'] * stk_ret.reindex(weights['static_capm_mvo'].index).fillna(0)).sum())
            all_returns['static_capm_mvo'].append({'date': date, 'ret': r_sc})

            # --- Kalman MVO (monthly) ---
            if date.day <= 5 and kalman_betas is not None and len(pw) >= 60:
                kb    = (kalman_betas.loc[prev_date].reindex(cur_tkrs).fillna(1.0)
                         if prev_date in kalman_betas.index else cur_betas)
                new_w = optimise(pw, kb, 'max_sharpe_beta_constrained')
                cost  = transaction_cost(weights['kalman_mvo'].reindex(cur_tkrs).fillna(0), new_w)
                weights['kalman_mvo'] = new_w
                all_returns['kalman_mvo'].append({'date': date, 'ret': -cost})

            r_km = float((weights['kalman_mvo'] * stk_ret.reindex(weights['kalman_mvo'].index).fillna(0)).sum())
            all_returns['kalman_mvo'].append({'date': date, 'ret': r_km})

        else:
            # Hold days — just compute portfolio return
            for s in ['adaptive_beta', 'static_capm_mvo', 'kalman_mvo']:
                r = float((weights[s] * stk_ret.reindex(weights[s].index).fillna(0)).sum())
                all_returns[s].append({'date': date, 'ret': r})

    print(f'  Fold complete — {len(test_idx)-1} trading days')

print('\n✅ Walk-forward backtest complete!')

In [ ]:
# Convert lists to Series, deduplicate dates (take last value per date)
returns_dict = {}
for strategy, records in all_returns.items():
    if not records:
        continue
    df = pd.DataFrame(records).set_index('date')['ret']
    df = df.groupby(df.index).last()  # deduplicate: keep last entry per date
    df = df.sort_index()
    returns_dict[strategy] = df
    print(f'{strategy}: {len(df)} days, annualised return = {df.mean()*252*100:.1f}%')

# Save individual returns CSVs
for name, series in returns_dict.items():
    series.to_csv(f'{ROOT}/results/{name}_returns.csv', header=['returns'])

rebalance_df = pd.DataFrame(rebalance_log)
if not rebalance_df.empty:
    rebalance_df.to_csv(f'{ROOT}/results/rebalance_log.csv', index=False)
    print(f'\nRebalance events: {len(rebalance_df)}')
    print(rebalance_df['signal'].value_counts())

## 4.5 — Performance Metrics

In [ ]:
def compute_metrics(returns, name=''):
    r = returns.dropna()
    ann_ret = r.mean() * 252
    ann_vol = r.std() * np.sqrt(252)
    sharpe  = ann_ret / ann_vol if ann_vol > 0 else 0
    downside = r[r < 0].std() * np.sqrt(252)
    sortino  = ann_ret / downside if downside > 0 else 0
    cum      = np.exp(r.cumsum())
    max_dd   = ((cum / cum.cummax()) - 1).min()
    n_years  = len(r) / 252
    cagr     = (np.exp(r.sum()))**(1/n_years) - 1 if n_years > 0 else 0
    calmar   = cagr / abs(max_dd) if max_dd != 0 else 0
    return {
        'CAGR (%)': round(cagr*100, 2),
        'Annual Return (%)': round(ann_ret*100, 2),
        'Annual Vol (%)': round(ann_vol*100, 2),
        'Sharpe Ratio': round(sharpe, 3),
        'Sortino Ratio': round(sortino, 3),
        'Max Drawdown (%)': round(max_dd*100, 2),
        'Calmar Ratio': round(calmar, 3),
    }

metrics_rows = {}
for name, returns in returns_dict.items():
    metrics_rows[name] = compute_metrics(returns, name)

metrics_df = pd.DataFrame(metrics_rows).T
metrics_df.to_csv(f'{ROOT}/results/performance_metrics.csv')
print('\n=== STRATEGY COMPARISON ===')
print(metrics_df.to_string())

## 4.6 — Equity Curves

In [ ]:
PALETTE = {
    'adaptive_beta':    '#1D9E75',
    'static_capm_mvo':  '#7F77DD',
    'kalman_mvo':       '#EF9F27',
    'equal_weight':     '#888780',
    'buy_hold_nifty':   '#D85A30',
}

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), gridspec_kw={'height_ratios': [3, 1]})

for name, returns in returns_dict.items():
    cum = np.exp(returns.cumsum()) * 100
    color = PALETTE.get(name, '#AAAAAA')
    lw    = 2.5 if name == 'adaptive_beta' else 1.5
    label = name.replace('_', ' ').title()
    ax1.plot(cum.index, cum.values, label=label, color=color, lw=lw)

# Shade COVID crash
ax1.axvspan('2020-02-01', '2020-05-01', alpha=0.15, color='red', label='COVID Crash')
ax1.axvspan('2023-01-25', '2023-03-15', alpha=0.15, color='orange', label='ADANI Crisis')
ax1.set_title('Cumulative Returns — Walk-Forward Out-of-Sample (base=100)', fontsize=13)
ax1.set_ylabel('Portfolio Value')
ax1.legend(loc='upper left', fontsize=9)

# AdaptiveBeta drawdown
if 'adaptive_beta' in returns_dict:
    r_ab = returns_dict['adaptive_beta']
    dd   = (np.exp(r_ab.cumsum()) / np.exp(r_ab.cumsum()).cummax() - 1) * 100
    ax2.fill_between(dd.index, dd.values, 0, alpha=0.4, color='#1D9E75')
    ax2.plot(dd.index, dd.values, color='#1D9E75', lw=1)
    ax2.set_ylabel('AdaptiveBeta Drawdown (%)')

plt.tight_layout()
plt.savefig(f'{ROOT}/results/equity_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: equity_curves.png')

## 4.7 — Stress Event Analysis

In [ ]:
STRESS_EVENTS = {
    'COVID Crash':   ('2020-02-01', '2020-05-01'),
    'ADANI Crisis':  ('2023-01-25', '2023-03-15'),
    '2018 Rate Hike':('2018-09-01', '2018-11-30'),
    'IL&FS Crisis':  ('2018-08-01', '2018-10-31'),
}

print('\n=== STRESS EVENT ANALYSIS ===')
all_stress_dfs = {}
for event, (start, end) in STRESS_EVENTS.items():
    rows = {}
    for name, returns in returns_dict.items():
        r = returns.loc[start:end]
        if len(r) == 0: continue
        cum  = float(np.exp(r.sum()) - 1) * 100
        dd_s = (np.exp(r.cumsum()) / np.exp(r.cumsum()).cummax() - 1)
        mdd  = float(dd_s.min()) * 100
        rows[name] = {'Cum Return (%)': round(cum, 2), 'Max DD (%)': round(mdd, 2)}
    stress_df = pd.DataFrame(rows).T
    all_stress_dfs[event] = stress_df
    print(f'\n{event} ({start} → {end}):')
    print(stress_df.to_string())
    stress_df.to_csv(f'{ROOT}/results/stress_{event.replace(" ","_")}.csv')

print('\nStress event CSVs saved.')

## 4.8 — QuantStats Tearsheet

In [ ]:
try:
    import quantstats as qs
    if 'adaptive_beta' in returns_dict and 'buy_hold_nifty' in returns_dict:
        qs.reports.html(
            returns_dict['adaptive_beta'],
            benchmark=returns_dict['buy_hold_nifty'],
            output=f'{ROOT}/results/adaptive_beta_tearsheet.html',
            title='AdaptiveBeta Strategy — M.Tech Thesis, SIT Pune',
        )
        print('QuantStats tearsheet saved: adaptive_beta_tearsheet.html')
except Exception as e:
    print(f'QuantStats error: {e}')

## 4.9 — Ablation Studies

In [ ]:
# Ablation: measure contribution of each component
# We use the test period returns to estimate impact of removing each component

print('=== ABLATION STUDY ===')
print('Component contributions to AdaptiveBeta performance\n')

if 'adaptive_beta' in metrics_df.index:
    base_sharpe = metrics_df.loc['adaptive_beta', 'Sharpe Ratio']
    print(f'Full AdaptiveBeta         Sharpe: {base_sharpe:.3f}')

    if 'static_capm_mvo' in metrics_df.index:
        no_lstm = metrics_df.loc['static_capm_mvo', 'Sharpe Ratio']
        print(f'Without LSTM (→ static)   Sharpe: {no_lstm:.3f}  (Δ = {base_sharpe - no_lstm:+.3f})')

    if 'equal_weight' in metrics_df.index:
        no_opt = metrics_df.loc['equal_weight', 'Sharpe Ratio']
        print(f'Without optimiser         Sharpe: {no_opt:.3f}  (Δ = {base_sharpe - no_opt:+.3f})')

    if 'buy_hold_nifty' in metrics_df.index:
        nifty_s = metrics_df.loc['buy_hold_nifty', 'Sharpe Ratio']
        print(f'Buy & Hold NIFTY50        Sharpe: {nifty_s:.3f}  (Δ = {base_sharpe - nifty_s:+.3f})')

# Rolling Sharpe comparison
fig, ax = plt.subplots(figsize=(14, 5))
for name, returns in returns_dict.items():
    roll_sharpe = (returns.rolling(252).mean() * 252) / (returns.rolling(252).std() * np.sqrt(252))
    ax.plot(roll_sharpe.index, roll_sharpe.values,
             label=name.replace('_',' ').title(), color=PALETTE.get(name,'#AAA'),
             lw=2.0 if name == 'adaptive_beta' else 1.2)
ax.axhline(0, color='white', lw=0.5, ls='--')
ax.set_title('Rolling 252-Day Sharpe Ratio')
ax.set_ylabel('Sharpe Ratio')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(f'{ROOT}/results/rolling_sharpe.png', dpi=120, bbox_inches='tight')
plt.show()

print('\n✅ Notebook 4 complete!')
print('All results saved to:', f'{ROOT}/results/')